# RAM Antivirus — Member 4: Cloud Posture, Vulnerabilities & Remediation
### **Assigned Scope:** C5 Misconfiguration | C10 Container/K8s | C12 Ransomware Precursor | C15 Supply Chain
**Author:** Member 4 — Cloud Posture & Vulnerability Management  
**Supporting Engines:** `cspm_engine.py`, `vulnerability_engine.py`, `remediation_engine.py`, `compliance_engine.py`

---
## Overview & Objectives
Member 4 enforces cloud security posture (CSPM), scans workload packages for known CVEs, executes automated dry-run remediations with rollback, and tracks ISO/SOC2/GDPR compliance.


In [ ]:
# %pip install -q pandas numpy matplotlib seaborn packaging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from packaging.version import parse as parse_version

sns.set_theme(style="whitegrid")
print("[OK] Member 4 workspace ready!")


---
## 1. CSPM (Cloud Security Posture Management) Rule Checks
Evaluates CIS AWS Foundations Benchmark checks:
- Public S3 buckets (`PutBucketAcl` public-read)
- Open security group ingress ports (`0.0.0.0/0` on 22/3389)
- Unencrypted RDS database instances


In [ ]:
cspm_checks = pd.DataFrame([
    {"resource": "s3://finance-data-bucket", "check": "Public Read Access", "status": "FAIL", "severity": "HIGH"},
    {"resource": "sg-01928374 (default)", "check": "Port 22 Open to 0.0.0.0/0", "status": "FAIL", "severity": "HIGH"},
    {"resource": "rds:mysql-db-prod", "check": "Storage Encryption Enabled", "status": "PASS", "severity": "INFO"},
    {"resource": "iam:root", "check": "MFA Enabled on Root Account", "status": "PASS", "severity": "INFO"},
])

print("CSPM Posture Scan Results:")
cspm_checks


---
## 2. Workload CVE Vulnerability Matching & CVSS Scoring
Scans `requirements.txt` packages and matches against local curated CVE advisory database.


In [ ]:
# Simulated CVE advisory database
advisories = [
    {"package": "requests", "cve": "CVE-2023-32681", "affected_below": "2.31.0", "cvss": 7.5, "severity": "High"},
    {"package": "urllib3", "cve": "CVE-2023-45803", "affected_below": "2.0.7", "cvss": 6.8, "severity": "Medium"},
    {"package": "jinja2", "cve": "CVE-2024-22195", "affected_below": "3.1.3", "cvss": 8.1, "severity": "High"}
]

installed_packages = {"requests": "2.28.1", "urllib3": "1.26.15", "jinja2": "3.1.4"}

matches = []
for adv in advisories:
    pkg = adv["package"]
    if pkg in installed_packages:
        inst_v = parse_version(installed_packages[pkg])
        aff_v = parse_version(adv["affected_below"])
        if inst_v < aff_v:
            matches.append({"package": pkg, "installed": installed_packages[pkg], **adv})

print("Vulnerability Scan Findings:")
pd.DataFrame(matches)


---
## 3. Automated Remediation Playbooks & Rollback Persistence
- Remediation operates in **dry-run mode**, surviving reboots via SQLite `remediation_history` persistence.


In [ ]:
playbooks = pd.DataFrame([
    {"playbook_id": "PB-S3-001", "trigger": "Public S3 Bucket", "action": "PutPublicAccessBlock", "mode": "DRY_RUN"},
    {"playbook_id": "PB-SG-002", "trigger": "Open Port 22", "action": "RevokeSecurityGroupIngress", "mode": "DRY_RUN"},
    {"playbook_id": "PB-EC2-003", "trigger": "Cryptomining Binary", "action": "StopInstance", "mode": "DRY_RUN"}
])

print("Remediation Playbook Catalog:")
playbooks


---
## 4. Compliance Framework Mapping (ISO 27001, SOC 2, GDPR)


In [ ]:
compliance_scores = {
    "ISO 27001:2022": 82.5,
    "SOC 2 Type II": 78.0,
    "GDPR Article 32": 85.0
}

plt.figure(figsize=(7, 3.8))
plt.bar(compliance_scores.keys(), compliance_scores.values(), color=["#1971c2", "#2f9e44", "#fab005"], width=0.45)
plt.title("Regulatory Compliance Posture Scores (%)", fontweight="bold")
plt.ylim(0, 100)
plt.ylabel("Compliance Score (%)")
for idx, (k, v) in enumerate(compliance_scores.items()):
    plt.annotate(f"{v}%", (idx, v / 2), ha="center", va="center", color="white", fontweight="bold")
plt.tight_layout()
plt.show()


---
## 5. Member 4 Viva Cheat Sheet
- **Q1: Does remediation modify real AWS production resources?**  
  *Ans:* No, remediation runs in dry-run mode generating verified Boto3 playbooks with durable SQLite rollback history.
- **Q2: What is CVSS?**  
  *Ans:* Common Vulnerability Scoring System (0.0 to 10.0 scale) measuring exploitability and severity.
- **Q3: CSPM vs Vulnerability scanning?**  
  *Ans:* CSPM checks cloud control-plane configurations (S3, SG, IAM); vulnerability scanning checks OS/package CVEs inside workloads.
